In [4]:
from PIL import Image
import math
import MiniNumPy as mnp
array = mnp.array

In [5]:
# load image and convert to 3D matrix (h, w, 3)
def load_image(path):
    img = Image.open(path).convert("RGB")
    w, h = img.size
    pixels = list(img.getdata())

    matrix = []
    for i in range(h):
        row = []
        for j in range(w):
            row.append(list(pixels[i*w + j]))  # [R, G, B]
        matrix.append(row)

    return array(matrix)

# save 3D matrix as image
def save_image(mat, path):
    h, w = mat.shape[0], mat.shape[1]

    img = Image.new("RGB", (w, h))
    flat = []

    for i in range(h):
        for j in range(w):
            pixel = mat.data[i][j]
            pixel = tuple(int(x) for x in pixel) 
            flat.append(pixel)

    img.putdata(flat)
    img.save(path)
    
# convert image to grayscale    
def to_grayscale(img):
    H, W, _ = img.shape
    gray = []

    for i in range(H):
        row = []
        for j in range(W):
            R, G, B = img.data[i][j]
            # gray value
            g = 0.299*R + 0.587*G + 0.114*B
            row.append([g, g, g])
        gray.append(row)

    return array(gray)


def apply_transform(points, M):
    result = []
    for p in points.data:
        v = array([[p[0]], [p[1]]])
        new = M @ v
        result.append([new.data[0][0], new.data[1][0]])
    return array(result)


def scale_matrix(sx, sy):
    return array([
        [sx, 0],
        [0, sy]
    ])

def rotation_matrix(theta):
    return array([
        [math.cos(theta), -math.sin(theta)],
        [math.sin(theta),  math.cos(theta)]
    ])

def shear_matrix(k):
    return array([
        [1, k],
        [0, 1]
    ])

def rotate_image(img, theta):
    H, W, _ = img.shape
    cx, cy = W // 2, H // 2  # picture center cordinates

    R = rotation_matrix(theta)

    new_img = []
    for y in range(H):
        row = []
        for x in range(W):

            # coordinates relative to center
            v = array([[x - cx], [y - cy]])
            new = R @ v

            nx = int(new.data[0][0] + cx)
            ny = int(new.data[1][0] + cy)

            # Assign pixel value
            if 0 <= nx < W and 0 <= ny < H:
                row.append(img.data[ny][nx])
            else:
                row.append([0, 0, 0])
        new_img.append(row)

    return array(new_img)


## Demo

In [6]:
# --- Load image ---
img = load_image("hawksbill_sea_turtle.jpg") 
print("Loaded image shape:", img.shape)

# --- Grayscale ---
gray = to_grayscale(img)
save_image(gray, "gray_image.jpg")
print("Saved grayscale image: gray_image.jpg")

# --- Rotate image 30 degrees ---
rot = rotate_image(img, math.radians(30))
save_image(rot, "rotated_30deg.jpg")
print("Saved rotated image: rotated_30deg.jpg")

# --- Demo 2D transformations ---
points = array([
    [0, 0],
    [1, 0],
    [1, 1],
    [0, 1]
])

S = scale_matrix(2, 2)
R = rotation_matrix(math.radians(45))
H = shear_matrix(1.0)

print("\nOriginal:", points)
print("Scaled:", apply_transform(points, S))
print("Rotated 45°:", apply_transform(points, R))
print("Sheared:", apply_transform(points, H))


Loaded image shape: (500, 750, 3)
Saved grayscale image: gray_image.jpg
Saved rotated image: rotated_30deg.jpg

Original: [[0 0]
 [1 0]
 [1 1]
 [0 1]]
Scaled: [[0 0]
 [2 0]
 [2 2]
 [0 2]]
Rotated 45°: [[0.0 0.0]
 [0.7071067811865476 0.7071067811865475]
 [1.1102230246251565e-16 1.414213562373095]
 [-0.7071067811865475 0.7071067811865476]]
Sheared: [[0.0 0]
 [1.0 0]
 [2.0 1]
 [1.0 1]]
